# Test RAGAS

In [10]:
import sys
import importlib
sys.path.insert(0, '/home/local/QCRI/fdeniz/projects/sspbench')


from sspbench.novelty.llm_utils import create_model_from_config
from sspbench.novelty.ragas_utils import (
    is_ragas_available,
    generate_qa_with_ragas,
    evaluate_qa_faithfulness
)

from sspbench.novelty.ragas_utils import SentenceTransformerEmbeddings


importlib.reload(sys.modules['sspbench.novelty.ragas_utils'])
from sspbench.novelty.ragas_utils import (
    is_ragas_available,
    generate_qa_with_ragas,
    evaluate_qa_faithfulness
)

print("✓ Imports successful")
print(f"RAGAS available: {is_ragas_available()}")

✓ Imports successful
RAGAS available: True


## Setup Eval Model

In [11]:
eval_config = {
    "type": "openai",
    "model": "gpt-oss",
    "api_url": "http://10.4.8.217:8000/v1",
    "api_token": "abc123",
    "api_version": "2024-12-01-preview"
}

try:
    eval_model = create_model_from_config(eval_config)
    print(f"✓ eval_model created successfully: {type(eval_model)}. Sample response: {eval_model.generate('Hello')}")
except Exception as e:
    print(f"✗ Failed to create eval_model: {e}")
    eval_model = None

embedding_model = SentenceTransformerEmbeddings("all-MiniLM-L6-v2")


Using cached model for config: gpt-oss
Generating with messages: [{'role': 'system', 'content': 'You are a helpful assistant.'}, {'role': 'user', 'content': 'Hello'}]
gpt-oss: Hello! How can I assist you today?
✓ eval_model created successfully: <class 'models.openai_model.OpenaiLLM'>. Sample response: ['Hello! How can I assist you today?']


In [12]:
# Sample paragraph for testing
test_paragraph = """
The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. 
It is named after the engineer Gustave Eiffel, whose company designed and built the tower. 
Constructed from 1887 to 1889 as the entrance arch to the 1889 World's Fair, it was initially 
criticized by some of France's leading artists and intellectuals for its design, but it has 
become a global cultural icon of France and one of the most recognizable structures in the world.
""".strip()

print("Test paragraph:")
print(test_paragraph)
print("\n" + "="*80 + "\n")

Test paragraph:
The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. 
It is named after the engineer Gustave Eiffel, whose company designed and built the tower. 
Constructed from 1887 to 1889 as the entrance arch to the 1889 World's Fair, it was initially 
criticized by some of France's leading artists and intellectuals for its design, but it has 
become a global cultural icon of France and one of the most recognizable structures in the world.




In [13]:
if eval_model and is_ragas_available():
    print("Generating Q&A pairs with RAGAS...\n")
    try:
        qa_pairs = generate_qa_with_ragas(
            paragraph=test_paragraph,
            agent_info=eval_model,
            embedding_model=embedding_model,
            num_questions=3
        )
        
        print(f"✓ Generated {len(qa_pairs)} Q&A pairs:\n")
        for i, qa in enumerate(qa_pairs, 1):
            print(f"Q{i}: {qa['question']}")
            print(f"A{i}: {qa['answer']}")
            print(f"Difficulty: {qa['difficulty']}")
            print("-" * 80)
    except Exception as e:
        print(f"✗ Error generating Q&A: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⚠️ Skipping test - eval_model or RAGAS not available")

Generating Q&A pairs with RAGAS...



Applying SummaryExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Generating with messages: [{'role': 'system', 'content': 'You are a helpful assistant.'}, {'role': 'user', 'content': 'Summarize the given text in less than 10 sentences.\nPlease return the output in a JSON format that complies with the following schema as specified in JSON Schema:\n{"properties": {"text": {"title": "Text", "type": "string"}}, "required": ["text"], "title": "StringIO", "type": "object"}Do not use single quotes in your response but double quotes,properly escaped with a backslash.\n\n--------EXAMPLES-----------\nExample 1\nInput: {\n    "text": "Artificial intelligence\\n\\nArtificial intelligence is transforming various industries by automating tasks that previously required human intelligence. From healthcare to finance, AI is being used to analyze vast amounts of data quickly and accurately. This technology is also driving innovations in areas like self-driving cars and personalized recommendations."\n}\nOutput: {\n    "text": "AI is revolutionizing industries by auto

Applying CustomNodeFilter:   0%|          | 0/1 [00:00<?, ?it/s]

Generating with messages: [{'role': 'system', 'content': 'You are a helpful assistant.'}, {'role': 'user', 'content': 'Given a document summary and node content, score the content of the node in 1 to 5 range.\nPlease return the output in a JSON format that complies with the following schema as specified in JSON Schema:\n{"properties": {"score": {"description": "1 to 5 score", "title": "Score", "type": "integer"}}, "required": ["score"], "title": "QuestionPotentialOutput", "type": "object"}Do not use single quotes in your response but double quotes,properly escaped with a backslash.\n\n-----------------------------\n\nNow perform the same with the following input\ninput: {\n    "document_summary": "The Eiffel Tower, a wrought‑iron lattice tower built for the 1889 World’s Fair and named after engineer Gustave Eiffel, was once criticized but has become a global cultural icon and one of the world’s most recognizable structures.",\n    "node_content": "The Eiffel Tower is a wrought-iron lat

...

Generating Scenarios:   0%|          | 0/1 [00:00<?, ?it/s]

Generating with messages: [{'role': 'system', 'content': 'You are a helpful assistant.'}, {'role': 'user', 'content': 'Given a list of themes and personas with their roles, associate each persona with relevant themes based on their role description.\nPlease return the output in a JSON format that complies with the following schema as specified in JSON Schema:\n{"properties": {"mapping": {"additionalProperties": {"items": {"type": "string"}, "type": "array"}, "title": "Mapping", "type": "object"}}, "required": ["mapping"], "title": "PersonaThemesMapping", "type": "object"}Do not use single quotes in your response but double quotes,properly escaped with a backslash.\n\n--------EXAMPLES-----------\nExample 1\nInput: {\n    "themes": [\n        "Empathy",\n        "Inclusivity",\n        "Remote work"\n    ],\n    "personas": [\n        {\n            "name": "HR Manager",\n            "role_description": "Focuses on inclusivity and employee support."\n        },\n        {\n            "n

Generating Samples:   0%|          | 0/3 [00:00<?, ?it/s]

Generating with messages: [{'role': 'system', 'content': 'You are a helpful assistant.'}, {'role': 'user', 'content': 'Generate a single-hop query and answer based on the specified conditions (persona, term, style, length) and the provided context. Ensure the answer is entirely faithful to the context, using only the information directly from the provided context.### Instructions:\n1. **Generate a Query**: Based on the context, persona, term, style, and length, create a question that aligns with the persona\'s perspective and incorporates the term.\n2. **Generate an Answer**: Using only the content from the provided context, construct a detailed answer to the query. Do not add any information not included in or inferable from the context.\n3. **Additional Context** (if provided): If llm_context is provided, use it as guidance for what type of question to generate (e.g., comparison questions, how-to questions, application-based questions) and how to structure the answer accordingly. Sti

Traceback (most recent call last):
  File "/home/local/QCRI/fdeniz/tmp/ipykernel_3755585/2830697676.py", line 4, in <module>
    qa_pairs = generate_qa_with_ragas(
  File "/home/local/QCRI/fdeniz/projects/sspbench/sspbench/novelty/ragas_utils.py", line 210, in generate_qa_with_ragas
    raise AttributeError(f"Could not find question attribute in {type(sample)}")
AttributeError: Could not find question attribute in <class 'ragas.testset.synthesizers.testset_schema.TestsetSample'>


In [ ]:
# Reload the module to pick up the fix
import importlib
importlib.reload(sys.modules['sspbench.novelty.ragas_utils'])
from sspbench.novelty.ragas_utils import (
    is_ragas_available,
    generate_qa_with_ragas,
    evaluate_qa_faithfulness
)
print("✓ Module reloaded with StringPromptValue support")

: 

: 

: 

: 

: 

## Test 2: Evaluate Q&A Faithfulness

Test the `evaluate_qa_faithfulness` function

In [ ]:
# Sample Q&A for evaluation
test_question = "Who designed the Eiffel Tower?"
test_answer = "Gustave Eiffel's company designed and built the tower."
test_context = test_paragraph

print("Test evaluation data:")
print(f"Question: {test_question}")
print(f"Answer: {test_answer}")
print(f"Context: {test_context[:100]}...")
print("\n" + "="*80 + "\n")

: 

: 

: 

: 

In [ ]:
if eval_model and is_ragas_available():
    print("Evaluating Q&A faithfulness with RAGAS...\n")
    try:
        scores = evaluate_qa_faithfulness(
            question=test_question,
            answer=test_answer,
            context=test_context,
            eval_model=eval_model
        )
        
        print("✓ Evaluation scores:")
        print(f"  Faithfulness: {scores['faithfulness']:.4f}")
        print(f"  Answerability (Answer Relevancy): {scores['answerability']:.4f}")
    except Exception as e:
        print(f"✗ Error evaluating Q&A: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⚠️ Skipping test - eval_model or RAGAS not available")

: 

: 

: 

: 

## Test 3: Full Workflow - Generate and Evaluate

Test the complete workflow of generating Q&A pairs and evaluating them

In [ ]:
if eval_model and is_ragas_available():
    print("Running full workflow: Generate + Evaluate\n")
    print("="*80)
    
    try:
        # Generate Q&A pairs
        print("\n1. Generating Q&A pairs...")
        qa_pairs = generate_qa_with_ragas(
            paragraph=test_paragraph,
            agent_info=eval_model,
            num_questions=2  # Use fewer for faster testing
        )
        print(f"   ✓ Generated {len(qa_pairs)} pairs\n")
        
        # Evaluate each pair
        print("2. Evaluating each Q&A pair...\n")
        for i, qa in enumerate(qa_pairs, 1):
            print(f"   Pair {i}:")
            print(f"   Q: {qa['question']}")
            print(f"   A: {qa['answer']}")
            
            scores = evaluate_qa_faithfulness(
                question=qa['question'],
                answer=qa['answer'],
                context=test_paragraph,
                eval_model=eval_model
            )
            
            print(f"   Faithfulness: {scores['faithfulness']:.4f}")
            print(f"   Answerability: {scores['answerability']:.4f}")
            print("-" * 80)
        
        print("\n✓ Full workflow completed successfully!")
        
    except Exception as e:
        print(f"\n✗ Error in workflow: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⚠️ Skipping test - eval_model or RAGAS not available")

: 

: 

: 

: 

## Summary

This notebook tested:
1. ✓ RAGAS availability check
2. ✓ Q&A generation with `generate_qa_with_ragas()`
3. ✓ Q&A evaluation with `evaluate_qa_faithfulness()`
4. ✓ Full workflow integration

All functions should work with RAGAS >= 0.2.0